# LLM-as-a-Judge

従来は人間の専門家がになっていた「評価」の役割をLLMに任せようという試み。



## 定義

LLM-as-a-Judgeは **定義済みのルール・基準・選好に基づいて、対象を評価するためにLLMを用いること** 。  
サーベイ論文 [A Survey on LLM-as-a-Judge](https://arxiv.org/abs/2411.15594) では次のように数式を使って定義している。


$$
\mathcal{E} \leftarrow \mathcal{P}_{LLM}(x \oplus C)
$$

- $\mathcal{E}$：LLM-as-a-Judge の一連のプロセス全体から、期待される形式で得られる最終的な評価。これは、スコア、選択、ラベル、文などであり得る。
- $\mathcal{P}_{LLM}$：対応する LLM によって定義される確率関数であり、生成は自己回帰的なプロセス。
- $x$：評価対象となる入力データであり、テキスト、画像、動画など、利用可能な任意の形式を取り得る。
- $C$：入力 $x$ に対するコンテキストであり、多くの場合、プロンプトテンプレートや対話履歴と組み合わされた情報。
- $\oplus$：入力 $x$ とコンテキスト $C$ を結合する演算子。この操作はコンテキストに応じて変わり、たとえば先頭・中央・末尾に配置するなどの形式を取り得る。


## 評価形式の代表パターン

1. スコアリング
    - 回答に対して、1〜5点、1〜10点、0〜100点などのスコアをつける形式。たとえば「正確性」「関連性」「詳細さ」「読みやすさ」をそれぞれ5段階で評価させる。
    - 長所は扱いやすいこと。短所は、LLMの点数感覚が不安定になりやすいこと。同じ品質差でも、あるときは7点と8点、別のときは5点と9点になることがある。
2. Yes/No判定
    - 「この回答は根拠文に忠実か」「この回答は質問に答えているか」などを、Yes/No、True/Falseで判定する形式
    - 単純で実装しやすいが、境界的なケースでは曖昧になりやすい。
3. ペアワイズ比較
    - 2つの回答A/Bを見せて、「どちらが良いか」を選ばせる形式
    - LLMは絶対点をつけるより、相対比較の方が安定しやすいことが多い。
4. 多肢選択
    - 複数の候補やラベルから最も適切なものを選ばせる形式。


## 信頼できる LLM as a Judgeを構築するには

サーベイ論文では

1. improvement
2. evaluation

の2軸に分けている

### 1. Improvement（改善）

#### 1.1 Prompt Design Strategy（プロンプト設計戦略）

##### 1. Improving LLMs’ Task Understanding（LLMのタスク理解の改善）

- Few-shot prompting：評価例をいくつか提示し、LLMが評価基準や出力の仕方を理解しやすくする方法
- Evaluation steps decomposition：評価を一度に行わせず、事実確認・関連性確認・総合判断など複数の手順に分解する方法
    - 例えばChain-of-Thought(CoT)
- Evaluation criteria decomposition：評価基準を「正確性」「網羅性」「明瞭性」などの細かい観点に分けて判定させる方法
    - 例えば「この回答を10点満点で評価してください。」ではなく、各観点ごとに点数を出させてから総合評価させる
- Shuffling contents：回答候補や評価対象の順序を入れ替え、位置バイアスなどの影響を抑える方法
- Conversion of evaluation tasks：スコア付けを比較問題に変えるなど、LLMが判断しやすい評価形式に変換する方法

##### 2. Standardizing LLMs’ Output Format（LLMの出力形式の標準化）

- Constraining outputs in structured formats：JSONや固定ラベルなど、後処理しやすい構造化形式で出力させる方法
- Providing evaluations with explanations：スコアやラベルだけでなく、その判断理由も併せて出力させる方法

#### 1.2 Capability Enhancement Strategy（能力拡張戦略）

##### 1. Specialized Fine-tuning（専用ファインチューニング）

- Evaluation template：評価対象・評価基準・出力形式を含むテンプレートを用いて、評価タスクに特化した学習を行う方法
- Deep transformation：単なるプロンプト調整ではなく、評価用データや評価形式を大きく変換して、judgeとしての能力を高める方法

##### 2. Feedback-Driven Iterative Refinement（フィードバック駆動の反復改善）

- INSTRUCTSCORE：LLMの評価結果や説明をフィードバックとして使い、評価の質を反復的に改善する手法。
- JADE：評価結果に対する自己修正やフィードバックを利用して、より信頼できるjudgeを構築する手法。
- Think-J：評価時に中間的な思考や検討過程を導入し、判断の精度を高める手法。

#### 1.3 Final Output Optimization Strategy（最終出力最適化戦略）

##### 1. Integrating Multi-Source Evaluation Results（複数ソースの評価結果の統合）

- Summarize by multiple rounds：複数回の評価結果をまとめることで、単発の評価に含まれる揺れを抑える方法
- Vote by multiple LLMs：複数のLLMに評価させ、多数決や集約によって最終判断を決める方法
- Hierarchical evaluation framework：大項目から小項目へ段階的に評価し、複雑な評価対象を構造的に判定する方法

##### 2. Direct Output Optimization（直接的な出力最適化）

- Score smoothing：スコアの極端なばらつきや不安定さを平滑化し、より安定した評価値に補正する方法
- Self validation：LLM自身に評価結果を再確認させ、矛盾や誤判定を検出・修正する方法


### 2. Evaluation（評価）

#### 2.1 Evaluation of Agreement with Human Judgments（人間判断との一致性評価）

- Agreement：LLM judgeの判断が人間評価とどの程度一致するかを単純な一致率などで測る指標。
- Cohen’s Kappa：偶然一致を考慮したうえで、LLM judgeと人間評価の一致度を測る指標。
- Spearman’s correlation：LLM judgeと人間評価の順位付けがどの程度一致しているかを測る順位相関指標。

#### 2.2 Evaluation of Bias（バイアス評価）

##### 1. Task-Agnostic Biases（タスク非依存バイアス）

- Diversity Bias：多様な表現や意見を不当に高く、または低く評価してしまう偏り。
- Cultural Bias：特定の文化・価値観・言語圏の前提に基づいて評価してしまう偏り。
- Self-Enhancement Bias：judge自身、または同系列のモデルが生成した回答を不当に高く評価してしまう偏り。

##### 2. Judgment-Specific Biases（判断固有バイアス）

- Position Bias：候補の提示順序によって、先頭または後方の回答を不当に選びやすくなる偏り。
- Compassion-fade Bias：評価対象が多数になるほど、個々の事例への共感や重みづけが弱くなる偏り。
- Style Bias：内容の正しさよりも、文体・丁寧さ・流暢さなどに引っ張られて評価してしまう偏り。
- Length Bias：長い回答を、内容の質とは無関係に「詳しい」「良い」と評価しやすい偏り。
- Concreteness Bias：具体例や詳細が多い回答を、必ずしも正確でなくても高く評価してしまう偏り。

#### 2.3 Evaluation of Adversarial Robustness（敵対的頑健性の評価）

- Adversarial Phrases Attack：評価を歪めるような誘導文や攻撃的フレーズを混ぜ、judgeが影響を受けるかを調べる攻撃。
- Null Model Attack：意味のない回答や空に近い回答を使って、judgeが誤って高評価しないかを調べる攻撃。
- Majority Opinions Attack：多数派意見を強調することで、judgeが内容ではなく多数派らしさに引っ張られるかを調べる攻撃。
- Meaningless Statement Robustness：無意味な文や関連性のない文に対して、judgeが適切に低評価できるかを確認する評価。

## 参考文献

- [A Survey on LLM-as-a-Judge](https://arxiv.org/abs/2411.15594) 
- [信頼できるLLM-as-a-Judgeの構築に向けた研究動向](https://zenn.dev/tsurubee/articles/1383f86eee4825)